# 📊 Data Collection & Feature Engineering Pipeline
---
## Overview
This notebook is the **first stage** of the stock valuation project.  
Its job is to:
1. Connect to **SimFin** (a financial data API) to pull real historical fundamentals (earnings, equity, debt, etc.)
2. Connect to **Yahoo Finance** (`yfinance`) to pull daily stock price history
3. Combine those two sources to engineer **13 financial features** per trading day, per stock
4. Apply the **Graham Intrinsic Value Formula** to automatically label each day as:
   - `0` = Undervalued  (price is more than 15% below the Graham fair value)
   - `1` = Fair Value    (price is within ±15% of Graham value)
   - `2` = Overvalued    (price is more than 15% above the Graham value)
5. Build a **balanced dataset** (equal number of samples per class) to prevent the model from being biased

> 💡 **Why balance the dataset?**  
> If 80% of all trading days are "Fair Value", a lazy model could achieve 80% accuracy by just always predicting "Fair".  
> Balancing forces the model to actually learn the difference between the three classes.

---
## Features Engineered

| Feature | Formula / Source | Why It Matters |
|---|---|---|
| `PE_Ratio` | Avg. Close Price / EPS | Expensive vs cheap relative to earnings |
| `ROE` | Net Income / Total Equity | How efficiently the company uses shareholder money |
| `ROA` | Net Income / Total Assets | How efficiently the company uses all its assets |
| `EPS` | Net Income / Shares Outstanding | Raw earnings per share |
| `Dividend_Yield` | Annual Dividend / Share Price | Income return; high yield can signal undervaluation |
| `Debt_to_Equity` | Long-Term Debt / Total Equity | Financial risk — higher D/E = more leveraged |
| `Price_to_Book` | Price / Book Value per Share | Classic Graham value metric |
| `Price_to_Sales` | Market Cap / Annual Revenue | Valuation relative to sales |
| `Revenue_Growth` | YoY revenue change (%) | Growth trajectory |
| `Operating_Margin` | Operating Income / Revenue | Core profitability |
| `Momentum` | 30-day MA / 90-day MA | Short-term price trend signal |
| `Volatility` | 30-day rolling std of daily returns | Risk / uncertainty measure |
| `sp500_close` | S&P 500 closing price | Market context / macro environment |

---
## The Graham Intrinsic Value Formula
> `Graham Value = √(22.5 × EPS × Book Value per Share)`

This formula was developed by Benjamin Graham, the father of value investing.  
The constant `22.5` assumes a maximum acceptable P/E of 15 and P/B of 1.5 (15 × 1.5 = 22.5).  
If the current stock price is significantly below this value, the stock may be a bargain.

## 1. Library Imports
We load all the external libraries needed for data collection, processing, and visualisation.

In [3]:
import simfin as sf        # SimFin API — historical financial statements
import os                  # File system operations (e.g., setting data directory)
import yfinance as yf      # Yahoo Finance — daily price history and basic company info
import pandas as pd        # DataFrames — the core data structure used throughout
import numpy as np         # Numerical operations (e.g., square root for Graham formula)
import time                # Adds small delays to avoid rate-limiting API calls
import matplotlib.pyplot as plt  # Visualisation — used at the end to display the feature table

## 2. SimFin API Configuration
SimFin provides access to annual income statements and balance sheets.  
- You need a **free API key** from [simfin.com](https://simfin.com)  
- Data is **downloaded once and cached locally** — subsequent runs use the local cache  
- `set_data_dir` tells SimFin where to save the cached files on your machine

> ⚠️ **Security Note:** Never commit your API key to GitHub. Consider storing it in an environment variable.

In [5]:
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("SIMFIN_API_KEY")

if not api_key:
    raise ValueError("API key not found")

sf.set_api_key(api_key)
sf.set_data_dir(os.path.expanduser('~/simfin_data/'))

print("SimFin configured")

SimFin configured


## 3. Sector Mapping & Loading SimFin Data

### 3a. Sector Mapping Dictionary
Yahoo Finance uses verbose sector names (e.g., `"Consumer Cyclical"`).  
We map them to shorter, consistent internal labels that are used when one-hot encoding sectors for the model.

### 3b. Loading Income & Balance Sheet Data
We load **two** SimFin datasets into memory:
- **Income Statement** (`sf_income`): Contains Revenue, Net Income, EPS, Operating Income
- **Balance Sheet** (`sf_balance`): Contains Total Equity, Total Assets, Long-Term Debt

These are indexed by `(Ticker, Date)`, allowing us to look up any company's financials for any year.

### 3c. `get_historical_fundamentals()` — Point-in-Time Lookup
This is the most important helper function. It answers the question:
> *"What were the actual financials for ticker X in year Y?"*

**Why is this important?**  
If we used today's EPS for all 5 years of historical data, we'd be introducing **look-ahead bias** —  
the model would be "cheating" by using future information to label past events.  
This function ensures each year uses only the data that was available at that time.

**Ratios computed inside this function:**

| Ratio | Calculation | Formula |
|---|---|---|
| EPS | Net Income ÷ Shares | Earnings per share |
| Book Value | Total Equity ÷ Shares | Net asset value per share |
| ROE | Net Income ÷ Total Equity | Return on equity |
| ROA | Net Income ÷ Total Assets | Return on assets |
| Operating Margin | Operating Income ÷ Revenue | Core profitability |
| Debt-to-Equity | Long-Term Debt ÷ Total Equity | Leverage ratio |

In [6]:
# ============================================================
# fetchStockInfo — SimFin backed (historical fundamentals per year)
#
# THE FIX: Instead of applying today's EPS/ROE/PE across all 5 years,
# we pull the ACTUAL fundamentals for each year from SimFin.
# This means 2021 rows use 2021 EPS, 2022 rows use 2022 EPS, etc.
# ============================================================

SECTOR_MAP = {
    'Technology':             'Technology',
    'Financial Services':     'Financials',
    'Healthcare':             'Healthcare',
    'Consumer Cyclical':      'Discretionary',
    'Consumer Defensive':     'Staples',
    'Energy':                 'Energy',
    'Industrials':            'Industrials',
    'Utilities':              'Utilities',
    'Real Estate':            'Utilities',
    'Communication Services': 'Technology',
    'Basic Materials':        'Industrials',
}

# Load SimFin datasets once — downloads and caches locally on first run
print('Loading SimFin datasets (first run downloads, then cached)...')
sf_income  = sf.load_income(variant='annual',  market='us')
sf_balance = sf.load_balance(variant='annual', market='us')
print(f'  Income rows:  {len(sf_income)}')
print(f'  Balance rows: {len(sf_balance)}')



def get_historical_fundamentals(ticker, year):
    """
    Return actual fundamentals for a ticker in a specific year from SimFin.
    Picks the most recent annual report on or before the target year.
    Returns a dict or None if data unavailable.
    """
    try:
        # ── Check ticker exists ───────────────────────────────────────────────
        all_tickers = sf_income.index.get_level_values('Ticker').unique()
        if ticker not in all_tickers:
            print(f"  [{ticker} {year}] SKIP — ticker not in SimFin income index")
            return None

        # ── Slice to this ticker ──────────────────────────────────────────────
        inc = sf_income.xs(ticker, level='Ticker').copy()
        bal = sf_balance.xs(ticker, level='Ticker').copy()


        # ── Convert index to datetime ─────────────────────────────────────────
        inc.index = pd.to_datetime(inc.index)
        bal.index = pd.to_datetime(bal.index)

        # ── Filter to years on or before target ───────────────────────────────
        inc_yr = inc[inc.index.year <= year]
        bal_yr = bal[bal.index.year <= year]
      

        if inc_yr.empty or bal_yr.empty:
            print(f"  [{ticker} {year}] SKIP — no data on or before {year}")
            return None

        # ── Take most recent row ──────────────────────────────────────────────
        ir = inc_yr.iloc[-1]
        br = bal_yr.iloc[-1]
        
        # ── Shares outstanding ────────────────────────────────────────────────
        shares = ir.get('Shares (Diluted)') or br.get('Shares (Diluted)')
      
        if not shares or shares == 0:
            print(f"  [{ticker} {year}] SKIP — shares outstanding is 0 or missing")
            return None

        # ── Raw fields ────────────────────────────────────────────────────────
        net_income = ir.get('Net Income')
        tot_equity = br.get('Total Equity')
        tot_assets = br.get('Total Assets')
        lt_debt    = br.get('Long Term Debt') or 0
        revenue    = ir.get('Revenue')
        op_income = ir.get('Operating Income (Loss)')

        




        # ── Computed ratios ───────────────────────────────────────────────────
        eps = (net_income / shares)    if net_income                             else None
        bv  = (tot_equity / shares)    if tot_equity                             else None
        roe = (net_income / tot_equity) if net_income and tot_equity and tot_equity != 0 else None
        roa = (net_income / tot_assets) if net_income and tot_assets             else None
        opm = (op_income  / revenue)   if op_income and revenue and revenue != 0 else None
        de  = (lt_debt    / tot_equity) if tot_equity and tot_equity != 0        else None



        # ── Check for missing critical fields ─────────────────────────────────
        missing = [k for k, v in {'eps': eps, 'de': de,'book_value': bv, 'roe': roe, 'roa': roa, 'op_margin': opm}.items() if v is None]
        if missing:
            print(f"  [{ticker} {year}] WARNING — missing computed fields: {missing}")

        return {
            'eps':       eps,
            'book_value': bv,
            'roe':       roe,
            'roa':       roa,
            'op_margin': opm,
            'de':        de,
            'revenue':   revenue,
        }

    except Exception as e:
        print(f"  [{ticker} {year}] ERROR — {type(e).__name__}: {e}")
        return None

def fetchStockInfo(ticker_symbol):
    """
    Builds a 5-year feature DataFrame where each year uses the
    ACTUAL historical fundamentals from that year (via SimFin).
    Falls back to current yfinance values only if SimFin has no data.
    """
    ticker_symbol = ticker_symbol.upper().strip()

    # Price history
    ticker_obj    = yf.Ticker(ticker_symbol)
    stock_history = ticker_obj.history(period='5y')
    if stock_history.empty:
        print(f'  Skipping {ticker_symbol}: no price history.')
        return None, 0

    time.sleep(0.3)
    info = ticker_obj.info

    # Sector from yfinance (does not change)
    raw_sector = info.get('sector')
    if not raw_sector:
        print(f'  Skipping {ticker_symbol}: no sector.')
        return None, 0
    sector = SECTOR_MAP.get(raw_sector)
    if not sector:
        print(f'  Skipping {ticker_symbol}: unmapped sector {raw_sector}.')
        return None, 0

    # SP500
    sp500 = yf.Ticker('^GSPC').history(period='5y')['Close']
    stock_history.index = stock_history.index.tz_convert(None)
    sp500.index         = sp500.index.tz_convert(None)
    sp500 = sp500.reindex(stock_history.index).ffill()

    # Build features year by year using historical fundamentals
    current_year  = pd.Timestamp.today().year
    yearly_frames = []

    for year in range(current_year - 4, current_year + 1):
        year_hist = stock_history[stock_history.index.year == year].copy()
        if year_hist.empty:
            continue

        # Get ACTUAL fundamentals for this year
        fund = get_historical_fundamentals(ticker_symbol, year)
        required_fields = ['eps', 'book_value', 'roe', 'roa', 'op_margin', 'de', 'revenue']
        if fund is None:
            print(f'  {ticker_symbol} {year}: ❌ No SimFin data → skipping ticker')
            return None, 0

        # Check missing fields
        missing = [f for f in required_fields if fund.get(f) in [None, 0]]
        if missing:
            print(f'  {ticker_symbol} {year}: ❌ Missing fields: {missing} → skipping ticker')
            return None, 0

        eps = fund['eps']
        bv  = fund['book_value']
        roe = fund['roe']
        roa = fund['roa']
        opm = fund['op_margin']
        de  = fund['de']
        revenue = fund['revenue']

        # Graham value
        gv = float(np.sqrt(22.5 * abs(eps * bv)))
        pe = year_hist['Close'].mean() / eps

        if pe == 0 or bv == 0 or revenue == 0:
            print(f'  {ticker_symbol} {year}: ❌ Invalid computed ratios → skipping ticker')
            return None, 0

        p2b = year_hist['Close'].mean() / bv
        p2s = info.get('marketCap') / revenue if info.get('marketCap') else None
        dy  = info.get('dividendYield') or 0.0
        rg  = info.get('revenueGrowth')

        if p2s is None or rg is None:
            print(f'  {ticker_symbol} {year}: ❌ Missing marketCap or revenueGrowth → skipping ticker')
            return None, 0

        def determine_label(row):
            price = row['Close']
            base = 0 if price < gv * 0.85 else (2 if price > gv * 1.15 else 1)
            if base == 2 and (roe > 0.25 and opm > 0.25 and pe < 50): return 1
            if base == 0 and (de > 5.0 and rg < -0.15): return 1
            return base

        df_y = year_hist.copy()
        df_y['Target']           = df_y.apply(determine_label, axis=1)
        df_y['PE_Ratio']         = pe
        df_y['ROE']              = roe
        df_y['ROA']              = roa
        df_y['EPS']              = eps
        df_y['Dividend_Yield']   = dy
        df_y['Debt_to_Equity']   = de
        df_y['Price_to_Book']    = p2b
        df_y['Price_to_Sales']   = p2s
        df_y['Revenue_Growth']   = rg
        df_y['Operating_Margin'] = opm
        df_y['Momentum']         = stock_history['Close'].rolling(30).mean() / stock_history['Close'].rolling(90).mean()
        df_y['Volatility']       = stock_history['Close'].pct_change().rolling(30).std()
        df_y['sp500_close']      = sp500.reindex(df_y.index).ffill()

        yearly_frames.append(df_y)

    if not yearly_frames:
        print(f'  Skipping {ticker_symbol}: no valid years.')
        return None, 0

    cols = [
        'PE_Ratio','ROE','ROA','EPS','Dividend_Yield','Debt_to_Equity',
        'Price_to_Book','Price_to_Sales','Revenue_Growth','Operating_Margin',
        'Momentum','Volatility','sp500_close','Target'
    ]

    df_final = pd.concat(yearly_frames)
    df_final = df_final[cols].dropna().iloc[90:]

    # Final Graham value (latest year only)
    latest = get_historical_fundamentals(ticker_symbol, current_year)

    if not latest or latest['eps'] in [None, 0] or latest['book_value'] in [None, 0]:
        print(f'  {ticker_symbol}: ❌ Missing latest fundamentals → skipping ticker')
        return None, 0

    gv_final = float(np.sqrt(22.5 * abs(latest['eps'] * latest['book_value'])))

    return df_final, gv_final


Loading SimFin datasets (first run downloads, then cached)...
Dataset "us-income-annual" on disk (2 days old).
- Loading from disk ... 

C:\Users\TheSa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\simfin\load.py:154: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  df = pd.read_csv(path, sep=';', header=0,


Done!
Dataset "us-balance-annual" on disk (2 days old).
- Loading from disk ... 

C:\Users\TheSa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\simfin\load.py:154: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  df = pd.read_csv(path, sep=';', header=0,


Done!
  Income rows:  16958
  Balance rows: 16958


## 4. Quick Sanity Test — Single Ticker
Before building the full dataset, we test the pipeline on a single well-known stock (AAPL).  
This verifies the API connections, the feature engineering, and the labelling function are all working correctly.  
If this returns a populated DataFrame, the pipeline is ready to scale up.

In [ ]:
data , grah = fetchStockInfo("aapl")

## 5. Building the Full Balanced Dataset

### 5a. The Stock Universe
We define a dictionary of sectors, each with ~20 tickers.  
The goal is **diversity** — covering different industries ensures the model learns patterns that generalise across all types of companies, not just technology or energy stocks.

### 5b. Per-Ticker Balancing (SAMPLES_PER_CLASS = 20)
For each ticker, we:
1. Fetch the full 5-year feature DataFrame (via `fetchStockInfo`)
2. Split rows into 3 groups by their `Target` label (0, 1, 2)
3. Sample up to 20 rows from each group

This **per-ticker cap** prevents any single highly-traded stock from dominating the dataset.

### 5c. Global Balancing (Anti-Bias Fix)
After collecting data from all tickers, we check the total size of each class.  
We then sample the **same number** from each class (equal to the smallest class).  
The result is a perfectly balanced 1:1:1 class ratio — critical for fair model training.

> 💡 **`sample(frac=1)`** at the end shuffles the final rows randomly.  
> This prevents the model from accidentally learning patterns based on row order.

In [ ]:
# 1. DEFINE THE UNIVERSE (10 per sector)
sector_map = {
    'Technology': [
        # Original
        'AAPL', 'MSFT', 'NVDA', 'ORCL', 'ADBE', 'CRM', 'INTC', 'CSCO', 'AMD', 'IBM',
        # Added
        'QCOM', 'TXN', 'NOW', 'AMAT', 'MU', 'LRCX', 'KLAC', 'SNPS', 'CDNS', 'HPQ'
    ],
    'Financials': [
    # Payments / Fintech (very stable)
    'V', 'MA', 'PYPL', 'SQ', 'FI', 'FIS',

    # Asset Management
    'BLK', 'TROW', 'IVZ',

    # Exchanges / Market Infrastructure
    'ICE', 'CME', 'NDAQ', 'CBOE',

    # Insurance (more compatible than banks)
    'MMC', 'BLK', 'SPGI', 'IVZ', 'TROW'
     ] ,
    'Healthcare': [
        # Original
        'PFE', 'JNJ', 'UNH', 'ABBV', 'MRK', 'LLY', 'AMGN', 'TMO', 'GILD', 'BMY',
        # Added
        'CVS', 'MDT', 'ABT', 'DHR', 'BSX', 'ISRG', 'EW', 'ZTS', 'REGN', 'VRTX'
    ],
    'Discretionary': [
        # Original
        'TSLA', 'AMZN', 'F', 'GM', 'HD', 'NKE', 'SBUX', 'MCD', 'BKNG', 'NCLH',
        # Added
        'LOW', 'TJX', 'EBAY', 'MAR', 'HLT', 'YUM', 'DRI', 'ROST', 'ORLY', 'DG'
    ],
    'Energy': [
        # Original
        'XOM', 'CVX', 'KMI', 'WMB', 'ENB', 'COP', 'SLB', 'EOG', 'MPC', 'VLO',
        # Added
        'PSX', 'EQT', 'HAL', 'BKR', 'KMI', 'DVN', 'HES', 'FANG', 'CTRA', 'APA'
    ],
    'Staples': [
        # Original
        'WMT', 'KO', 'PEP', 'COST', 'PG', 'EL', 'CL', 'MO', 'TGT', 'PM',
        # Added
        'MDLZ', 'GIS', 'KHC', 'HSY', 'SYY', 'ADM', 'BG', 'CAG', 'MKC', 'CHD'
    ],
    'Industrials': [
        # Original
        'BA', 'CAT', 'GE', 'UPS', 'FDX', 'ITW', 'LMT', 'MMM', 'DE', 'RTX',
        # Added
        'NOC', 'GD', 'EMR', 'ETN', 'PH', 'ROK', 'TT', 'CARR', 'OTIS', 'WAB'
    ],
    'Utilities': [
        # Original
        'NEE', 'DUK', 'SO', 'D', 'AEP', 'PLD', 'AMT', 'EQIX', 'CCI', 'O',
        # Added
        'EXC', 'XEL', 'WEC', 'ES', 'AWK', 'DTE', 'PPL', 'FE', 'ETR', 'AEE'
    ]
}
# --- RE-CHECKED BALANCING LOGIC ---
all_balanced_data = []
SAMPLES_PER_CLASS = 20 

for sector, tickers in sector_map.items():
    print(f"\n--- Balancing Sector: {sector} ---")
    for ticker in tickers:
        try:
            # 1. Fetch
            df_full, g_val = fetchStockInfo(ticker)
            
            # --- CRITICAL CHECK: Is there data? ---
            if df_full is None or df_full.empty:
                print(f"Skipping {ticker}: No data returned from fetcher.")
                continue

            # 2. Split classes
            c0 = df_full[df_full['Target'] == 0]
            c1 = df_full[df_full['Target'] == 1]
            c2 = df_full[df_full['Target'] == 2]

            # 3. Sample (only if class has data)
            s0 = c0.sample(n=min(len(c0), SAMPLES_PER_CLASS)) if not c0.empty else pd.DataFrame()
            s1 = c1.sample(n=min(len(c1), SAMPLES_PER_CLASS)) if not c1.empty else pd.DataFrame()
            s2 = c2.sample(n=min(len(c2), SAMPLES_PER_CLASS)) if not c2.empty else pd.DataFrame()

            # 4. Combine
            chunk = pd.concat([s0, s1, s2])
            if not chunk.empty:
                chunk['Sector'] = sector
                chunk['Ticker'] = ticker
                all_balanced_data.append(chunk)
                print(f"Collected {len(chunk)} samples for {ticker}")
            else:
                print(f"No samples found for {ticker} in any class.")

        except Exception as e:
            print(f"Skipping {ticker} due to error: {e}")

# --- STAGE 2: REDUCE TARGET 1 TO 400 ---
# --- STAGE 1: Combine everything collected in the loop ---
df_collected = pd.concat(all_balanced_data)

# --- STAGE 2: THE "ANTI-BIAS" FIX (The steps you provided) ---
# Find the smallest class size to determine our "cap"
min_size = min(len(df_collected[df_collected['Target']==0]), 
               len(df_collected[df_collected['Target']==1]), 
               len(df_collected[df_collected['Target']==2]))

print(f"Balancing dataset... Smallest class size is {min_size}")

# Sample that exact amount from each to create a perfect 1:1:1 ratio
final_df = pd.concat([
    df_collected[df_collected['Target']==0].sample(min_size, random_state=42),
    df_collected[df_collected['Target']==1].sample(min_size, random_state=42),
    df_collected[df_collected['Target']==2].sample(min_size, random_state=42)
]).sample(frac=1).reset_index(drop=True) # frac=1 shuffles the rows

# --- STAGE 3: Save and Verify ---
final_df.to_csv('balanced_stock_data.csv', index=False)

print("\n--- FINAL BALANCED COUNTS ---")
print(final_df['Target'].value_counts())

## 6. Visualising a Sample of the Feature Matrix
This cell fetches a fresh sample of AAPL data and displays the **5 most recent rows** as a formatted table.  
This gives a quick visual check that all 13 features have been correctly computed and the data looks sensible before passing it to the analysis or model notebooks.

> The `intrinsic_price` returned is the **Graham Intrinsic Value** — the theoretical fair price of the stock based on its most recent EPS and book value.

In [ ]:
data, intrinsic_price = fetchStockInfo("aapl")
table_data = data.tail(5).copy()
table_data = table_data.round(3)

# 2. Create the Plot
fig, ax = plt.subplots(figsize=(14, 4)) # Adjust size for your report
ax.axis('tight')
ax.axis('off')

# 3. Create the Table
table = ax.table(cellText=table_data.values, 
                 colLabels=table_data.columns, 
                 cellLoc='center', 
                 loc='center',
                 colColours=["#f2f2f2"] * len(table_data.columns)) # Light grey header

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.7) # Scale for better spacing

plt.title("Latest Stock Feature Matrix", fontsize=14, pad=20)
plt.show()